In [ ]:
# PyTorch ana kütüphanesini içe aktarır - tensor işlemleri ve derin öğrenme için
import torch

# PyTorch'un sinir ağı modülünü içe aktarır - katmanlar, aktivasyon fonksiyonları, kayıp fonksiyonları
import torch.nn as nn

# Stokastik Gradyan İnişi optimizasyonu için
from torch.optim import SGD

# PyTorch veri yükleme yardımcıları - Dataset ve DataLoader sınıfları
from torch.utils.data import Dataset, DataLoader

# Sayısal hesaplamalar ve dizi işlemleri için NumPy kütüphanesi
import numpy as np

# Grafik ve görselleştirme için matplotlib kütüphanesi
import matplotlib.pyplot as plt

# Eğitim/test verisi bölme fonksiyonu
from sklearn.model_selection import train_test_split

# Model performans değerlendirme metrikleri
from sklearn.metrics import accuracy_score

# MNIST veri setini okumak için özel kütüphane
import idx2numpy

In [ ]:
# MNIST veri seti dosya yolu
MNIST_DIR = "mnist/"

# MNIST eğitim görüntülerini oku (60000 adet 28x28 piksel görüntü)
# idx2numpy.convert_from_file(): IDX formatındaki dosyayı NumPy dizisine dönüştürür
X_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")

# Görüntüleri düzleştir ve normalize et: (60000, 28, 28) -> (60000, 784) ve [0,255] -> [0,1]
# reshape(60000, -1): -1 otomatik boyut hesaplama (28*28=784)
# /255.0: Piksel değerlerini 0-1 aralığına normalize et
X_mnist = X_mnist.reshape(60000, -1) / 255.0

# MNIST eğitim etiketlerini oku (0-9 arası rakam etiketleri)
y_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

# Veri setini eğitim ve test setlerine böl
# test_size=0.2: %20 test, %80 eğitim
# random_state=42: Tutarlı bölünme için seed
X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)  

# NumPy dizilerini PyTorch tensörlerine dönüştür
# astype(np.float32): 32-bit float veri tipi (GPU uyumluluğu için)
x = torch.from_numpy(X_train.astype(np.float32))

# Etiketleri int64 tipine dönüştür (CrossEntropyLoss için gerekli)
y = torch.from_numpy(y_train.astype(np.int64))

In [ ]:
# PyTorch Dataset sınıfından türetilen özel MNIST veri seti sınıfı
class MnistDataset(Dataset):
    def __init__(self, X, y):
        # Veri seti başlangıç fonksiyonu - veri yükleme ve ön işleme
        
        # NumPy dizilerini PyTorch tensörlerine dönüştür
        # astype(np.float32): Özellikler için 32-bit float veri tipi
        self.x = torch.from_numpy(X.astype(np.float32))
        
        # astype(np.int64): Etiketler için 64-bit integer veri tipi (sınıflandırma için)
        self.y = torch.from_numpy(y.astype(np.int64))

    def __getitem__(self, index):
        # Belirli bir indeksteki veri örneğini döndür
        # Bu metod, dataset[index] sözdiziminin çalışmasını sağlar
        # Dönen değer: (özellik_vektörü, etiket) çifti
        return self.x[index], self.y[index]

    def __len__(self):
        # Veri setindeki toplam örnek sayısını döndür
        # Bu metod, len(dataset) sözdiziminin çalışmasını sağlar
        # shape[0]: İlk boyut (örnek sayısı)
        return self.x.shape[0]

In [ ]:
# Genel model eğitim fonksiyonu - farklı mimarileri karşılaştırmak için
def train_model(model, train_data_loader, test_data_loader):
    # Çok sınıflı sınıflandırma için kayıp fonksiyonu
    # CrossEntropyLoss = Softmax + NegativeLogLikelihood birleşimi
    criterion = nn.CrossEntropyLoss()
    
    # Stokastik Gradyan İnişi optimizeri
    # model.parameters(): Modelin tüm öğrenilebilir parametrelerini al
    # lr=0.01: Öğrenme hızı (learning rate)
    optimizer = SGD(model.parameters(), lr=0.01)
    
    # Eğitim ve validasyon kayıplarını takip etmek için listeler
    train_loss = []      # Her epoch'taki ortalama eğitim kaybı
    validation_loss = [] # Her epoch'taki ortalama validasyon kaybı

    # 20 epoch boyunca eğitim
    for epoch in range(20):
        # Her epoch için eğitim kayıplarını topla
        batch_train_loss = []
        
        # Eğitim veri setindeki tüm batch'ler üzerinde döngü
        for X, y in train_data_loader:
            # İleri yayılım: Model tahminleri üret
            y_hat = model(X)
            
            # Kayıp hesapla: Tahmin vs gerçek etiketler
            loss = criterion(y_hat, y)
            
            # Geri yayılım: Gradyanları hesapla
            loss.backward()
            
            # Parametreleri güncelle
            optimizer.step()
            
            # Gradyanları sıfırla (bir sonraki iterasyon için)
            optimizer.zero_grad()
            
            # Bu batch'in kaybını kaydet
            batch_train_loss.append(loss.item())
        
        # Bu epoch'un ortalama eğitim kaybını hesapla ve kaydet
        train_loss.append(np.array(batch_train_loss).mean())

        # Validasyon aşaması (gradyan hesaplaması olmadan)
        with torch.no_grad():
            batch_validation_loss = []
            
            # Test veri setindeki tüm batch'ler üzerinde döngü
            for X, y in test_data_loader:
                # Modelden tahmin al
                y_hat = model(X)
                
                # Validasyon kaybını hesapla
                loss = criterion(y_hat, y)
                
                # Bu batch'in validasyon kaybını kaydet
                batch_validation_loss.append(loss.item())

            # Bu epoch'un ortalama validasyon kaybını hesapla ve kaydet
            validation_loss.append(np.array(batch_validation_loss).mean())
        
        # Her 5 epoch'ta ilerleme raporu yazdır
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} done!")
    
    # Eğitim ve validasyon kayıp listelerini döndür
    return train_loss, validation_loss

In [ ]:
# Model Mimarisi 1: En Basit Tek Katmanlı Lineer Model
class MNistClassifier_1(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Tek lineer katman: Doğrudan girdiden çıkışa
        # 784: Düzleştirilmiş 28x28 piksel görüntü
        # 10: MNIST'te 10 farklı rakam sınıfı (0-9)
        # Bu model sadece lineer dönüşüm yapar: y = Wx + b
        self.input_layer = nn.Linear(784, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu - sadece lineer dönüşüm
        # Hiçbir aktivasyon fonksiyonu yok (tamamen lineer)
        # Bu model sadece lineer ilişkileri öğrenebilir
        x = self.input_layer(x)
        return x

In [ ]:
# Model Mimarisi 2: Sigmoid Aktivasyonlu İki Katmanlı Model
class MNistClassifier_2(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Girdi katmanı: 784 nörondan 16 nörona lineer dönüşüm
        # 16: Gizli katman boyutu (hiperparametre - Model 1'e göre daha küçük)
        self.input_layer = nn.Linear(784, 16)
        
        # Sigmoid aktivasyon fonksiyonu
        # σ(x) = 1/(1+e^(-x)) - çıkışı (0,1) aralığında sıkıştırır
        # Avantajları: Türevi kolay hesaplanır, sürekli ve diferansiyel
        # Dezavantajları: Gradyan kaybı problemi, hesaplama maliyeti
        self.activation = nn.Sigmoid()
        
        # Çıkış katmanı: 16 nörondan 10 sınıfa lineer dönüşüm
        self.output_layer = nn.Linear(16, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu
        
        # Girdi katmanından geçir: 784 -> 16
        x = self.input_layer(x)
        
        # Sigmoid aktivasyonu uygula (non-linearity ekleme)
        # Bu sayede model non-lineer örüntüleri öğrenebilir
        x = self.activation(x)
        
        # Çıkış katmanından geçir: 16 -> 10
        x = self.output_layer(x)
        
        # Logit değerlerini döndür
        return x

In [ ]:
# Model Mimarisi 3: ReLU Aktivasyonlu İki Katmanlı Model (Modern Yaklaşım)
class MNistClassifier_3(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Girdi katmanı: 784 nörondan 16 nörona lineer dönüşüm
        # Model 2 ile aynı mimari ama farklı aktivasyon fonksiyonu
        self.input_layer = nn.Linear(784, 16)
        
        # ReLU aktivasyon fonksiyonu
        # ReLU(x) = max(0, x) - negatif değerleri sıfırlar, pozitif değerleri olduğu gibi bırakır
        # Avantajları: Hesaplama açısından çok verimli, gradyan kaybı problemi yok
        # Seyrek aktivasyon sağlar, derin ağlarda daha iyi performans
        self.activation = nn.ReLU()
        
        # Çıkış katmanı: 16 nörondan 10 sınıfa lineer dönüşüm
        self.output_layer = nn.Linear(16, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu
        
        # Girdi katmanından geçir: 784 -> 16
        x = self.input_layer(x)
        
        # ReLU aktivasyonu uygula
        # Bu, modern derin öğrenme uygulamalarında en yaygın kullanılan aktivasyon
        # Gradyan akışını iyileştirir ve eğitimi hızlandırır
        x = self.activation(x)
        
        # Çıkış katmanından geçir: 16 -> 10
        x = self.output_layer(x)
        
        # Logit değerlerini döndür
        return x

In [ ]:
# Eğitim ve test veri yükleyicilerini hazırla

# Eğitim veri seti için Dataset ve DataLoader oluştur
dataset_train = MnistDataset(X_train, y_train)
data_loader_train = DataLoader(
    dataset=dataset_train,   # MnistDataset örneği
    batch_size=256,          # Her batch'te 256 örnek (GPU belleği için optimize)
    shuffle=True             # Her epoch'ta veriyi karıştır (overfitting'i önler)
)

# Test veri seti için Dataset ve DataLoader oluştur
dataset_test = MnistDataset(X_test, y_test)
data_loader_test = DataLoader(
    dataset=dataset_test,    # Test veri seti
    batch_size=256,          # Aynı batch boyutu (tutarlılık için)
    shuffle=True             # Test için karıştırma zorunlu değil ama yapabilir
)

In [ ]:
# Kayıp grafiği çizim fonksiyonu - model performanslarını karşılaştırmak için
def plot_loss(train_loss, validation_loss):
    # Grafik boyutunu ayarla
    plt.figure(figsize=(8, 4))
    
    # Eğitim kaybını mavi renkte çiz
    # train_loss: Her epoch'taki ortalama eğitim kaybı listesi
    plt.plot(train_loss, c="b", label="Train")
    
    # Validasyon kaybını kırmızı renkte çiz
    # validation_loss: Her epoch'taki ortalama validasyon kaybı listesi
    plt.plot(validation_loss, c="r", label="Validation")
    
    # Lejantı göster (hangi çizginin neyi temsil ettiği)
    plt.legend()
    
    # Grafiği göster
    # Bu grafik sayesinde:
    # - Modelin öğrenme sürecini izleyebiliriz
    # - Overfitting durumunu tespit edebiliriz (validation loss artarken train loss azalıyorsa)
    # - Underfitting durumunu görebiliriz (her iki loss da yüksek kalıyorsa)
    plt.show()

In [ ]:
# Model 1 Eğitimi: Tek Katmanlı Lineer Model
# Bu en basit mimari - sadece lineer dönüşüm, aktivasyon fonksiyonu yok

# Model örneği oluştur
model_1 = MNistClassifier_1()

# Modeli eğit ve eğitim/validasyon kayıplarını al
# train_model fonksiyonu: 20 epoch boyunca eğitim yapar
# Dönen değerler: her epoch'taki ortalama train ve validation loss listeleri
train_loss, validation_loss = train_model(model_1, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Bu grafik Model 1'in öğrenme sürecini gösterir
# Beklenti: Yüksek bias (underfitting) nedeniyle sınırlı performans
plot_loss(train_loss, validation_loss)

In [ ]:
# Model 2 Eğitimi: Sigmoid Aktivasyonlu İki Katmanlı Model
# Non-linear aktivasyon eklemenin etkisini gözlemleyeceğiz

# Model örneği oluştur
model_2 = MNistClassifier_2()

# Modeli eğit ve sonuçları al
# Bu model gizli katman (16 nöron) ve sigmoid aktivasyon içerir
# Model 1'e göre daha yüksek model kapasitesi bekleniyor
train_loss, validation_loss = train_model(model_2, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Beklenti: Model 1'e göre daha düşük kayıp değerleri
# Ancak sigmoid'in gradyan kaybı problemi nedeniyle yavaş öğrenme olabilir
plot_loss(train_loss, validation_loss)

In [ ]:
# Model 3 Eğitimi: ReLU Aktivasyonlu İki Katmanlı Model
# Modern derin öğrenme yaklaşımı - ReLU aktivasyon fonksiyonu

# Model örneği oluştur
model_3 = MNistClassifier_3()

# Modeli eğit ve sonuçları al
# Bu model Model 2 ile aynı mimariye sahip ama ReLU aktivasyon kullanır
# ReLU'nun avantajları: hızlı hesaplama, iyi gradyan akışı
train_loss, validation_loss = train_model(model_3, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Beklenti: Model 2'ye göre daha hızlı yakınsama
# ReLU'nun gradyan akışı iyileştirmesi sayesinde daha iyi performans
plot_loss(train_loss, validation_loss)

In [ ]:
# Karşılaştırmalı Analiz İçin Duplike Import'lar
# (Bu hücre, notebook'un bağımsız çalışabilmesi için gerekli import'ları tekrarlar)

# PyTorch ana kütüphanesi
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import Dataset, DataLoader

# Sayısal hesaplamalar ve görselleştirme
import numpy as np
import matplotlib.pyplot as plt

# Model değerlendirme araçları
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# MNIST veri seti okuma kütüphanesi
import idx2numpy

In [ ]:
# Veri Hazırlığının Tekrarı (Notebook Bağımsızlığı İçin)

# MNIST veri seti yolunu belirle
MNIST_DIR = "mnist/"

# MNIST görüntülerini oku ve ön işle
# 60000 adet 28x28 piksel görüntü -> 60000x784 düzleştirilmiş format
X_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")
X_mnist = X_mnist.reshape(60000, -1) / 255.0  # Normalizasyon: [0,255] -> [0,1]

# MNIST etiketlerini oku (0-9 arası rakam etiketleri)
y_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

# Veri setini eğitim/test olarak böl (%80 eğitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)  

# PyTorch tensörlerine dönüştür (GPU uyumluluğu için)
x = torch.from_numpy(X_train.astype(np.float32))
y = torch.from_numpy(y_train.astype(np.int64))

In [ ]:
# Dataset Sınıfının Tekrarı (Modülerlik İçin)
class MnistDataset(Dataset):
    def __init__(self, X, y):
        # Veri başlatma: NumPy dizilerini PyTorch tensörlerine dönüştür
        self.x = torch.from_numpy(X.astype(np.float32))  # Özellikler
        self.y = torch.from_numpy(y.astype(np.int64))    # Etiketler

    def __getitem__(self, index):
        # Indeks bazlı veri erişimi: dataset[i] sözdizimini destekler
        return self.x[index], self.y[index]

    def __len__(self):
        # Veri seti uzunluğu: len(dataset) sözdizimini destekler
        return self.x.shape[0]

In [ ]:
# Eğitim Fonksiyonunun Tekrarı (Karşılaştırmalı Analiz İçin)
def train_model(model, train_data_loader, test_data_loader):
    # Eğitim bileşenlerini hazırla
    criterion = nn.CrossEntropyLoss()          # Çok sınıflı sınıflandırma kayıp fonksiyonu
    optimizer = SGD(model.parameters(), lr=0.01)  # SGD optimizeri

    # Kayıp takip listeleri
    train_loss = []      # Eğitim kayıpları
    validation_loss = [] # Validasyon kayıpları

    # 20 epoch eğitim döngüsü
    for epoch in range(20):
        # Eğitim fazı
        batch_train_loss = []
        for X, y in train_data_loader:
            y_hat = model(X)           # İleri yayılım
            loss = criterion(y_hat, y) # Kayıp hesaplama
            loss.backward()            # Geri yayılım
            optimizer.step()           # Parametre güncelleme
            optimizer.zero_grad()      # Gradyan sıfırlama
            batch_train_loss.append(loss.item())
        
        train_loss.append(np.array(batch_train_loss).mean())

        # Validasyon fazı (gradyan hesaplaması olmadan)
        with torch.no_grad():
            batch_validation_loss = []
            for X, y in test_data_loader:
                y_hat = model(X)
                loss = criterion(y_hat, y)
                batch_validation_loss.append(loss.item())
            validation_loss.append(np.array(batch_validation_loss).mean())
        
        # İlerleme raporu (her 5 epoch'ta)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} done!")
            
    return train_loss, validation_loss

In [ ]:
# Model Sınıflarının Tekrarı - Model 1: Lineer (Basit)
class MNistClassifier_1(nn.Module):
    def __init__(self):
        super().__init__()
        # Tek katmanlı doğrudan bağlantı: 784 -> 10
        # Hiçbir gizli katman ve aktivasyon fonksiyonu yok
        self.input_layer = nn.Linear(784, 10)
        
    def forward(self, x):
        # Sadece lineer dönüşüm: y = Wx + b
        # En basit model - sadece lineer ilişkileri öğrenebilir
        x = self.input_layer(x)
        return x

In [ ]:
# Model 2: Sigmoid Aktivasyonlu (Geleneksel)
class MNistClassifier_2(nn.Module):
    def __init__(self):
        super().__init__()
        # İki katmanlı mimari: 784 -> 16 -> 10
        self.input_layer = nn.Linear(784, 16)  # Gizli katman
        self.activation = nn.Sigmoid()         # Sigmoid aktivasyon
        self.output_layer = nn.Linear(16, 10)  # Çıkış katmanı
        
    def forward(self, x):
        # Katmanlı ileri yayılım:
        x = self.input_layer(x)    # 784 -> 16 lineer dönüşüm
        x = self.activation(x)     # Sigmoid aktivasyon (0,1) aralığı
        x = self.output_layer(x)   # 16 -> 10 lineer dönüşüm
        return x

In [ ]:
# Model 3: ReLU Aktivasyonlu (Modern)
class MNistClassifier_3(nn.Module):
    def __init__(self):
        super().__init__()
        # İki katmanlı mimari: 784 -> 16 -> 10 (Model 2 ile aynı)
        self.input_layer = nn.Linear(784, 16)  # Gizli katman
        self.activation = nn.ReLU()            # ReLU aktivasyon (modern seçim)
        self.output_layer = nn.Linear(16, 10)  # Çıkış katmanı
        
    def forward(self, x):
        # Katmanlı ileri yayılım:
        x = self.input_layer(x)    # 784 -> 16 lineer dönüşüm
        x = self.activation(x)     # ReLU aktivasyon max(0,x)
        x = self.output_layer(x)   # 16 -> 10 lineer dönüşüm
        return x

In [ ]:
# DataLoader'ların Tekrar Hazırlanması

# Eğitim veri yükleyici
dataset_train = MnistDataset(X_train, y_train)
data_loader_train = DataLoader(
    dataset=dataset_train, 
    batch_size=256,  # Mini-batch boyutu
    shuffle=True     # Her epoch'ta karıştır
)

# Test veri yükleyici
dataset_test = MnistDataset(X_test, y_test)
data_loader_test = DataLoader(
    dataset=dataset_test, 
    batch_size=256,  # Aynı batch boyutu
    shuffle=True     # Test için de karıştırma
)

In [ ]:
# Kayıp Görselleştirme Fonksiyonunun Tekrarı
def plot_loss(train_loss, validation_loss):
    # Grafik ayarları
    plt.figure(figsize=(8, 4))
    
    # Eğitim ve validasyon kayıplarını aynı grafikte göster
    plt.plot(train_loss, c="b", label="Train")        # Mavi: eğitim kaybı
    plt.plot(validation_loss, c="r", label="Validation")  # Kırmızı: validasyon kaybı
    
    # Lejant ve gösterim
    plt.legend()
    plt.show()
    
    # Bu grafik, modelin performansını değerlendirmek için kritik:
    # - Converge olma durumu
    # - Overfitting tespiti
    # - Model kapasitesi değerlendirmesi

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 1 (Tekrar)
# Lineer model performansını yeniden değerlendirme

# Yeni model örneği oluştur (parametreler rastgele başlatılır)
model_1 = MNistClassifier_1()

# Eğitim sürecini başlat ve kayıpları takip et
# Bu, en basit mimarinin referans performansını belirler
train_loss, validation_loss = train_model(model_1, data_loader_train, data_loader_test)

# Sonuçları görselleştir
# Lineer modelin sınırlarını gözlemleyeceğiz
plot_loss(train_loss, validation_loss)

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 2 (Tekrar)
# Sigmoid aktivasyonunun etkisini tekrar gözlemleme

# Yeni sigmoid model örneği
model_2 = MNistClassifier_2()

# Eğitim ve performans takibi
# Non-linear aktivasyonun Model 1'e göre iyileştirmesini bekleriz
train_loss, validation_loss = train_model(model_2, data_loader_train, data_loader_test)

# Sigmoid modelin öğrenme sürecini görselleştir
# Gradyan kaybı ve yakınsama hızını analiz edelim
plot_loss(train_loss, validation_loss)

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 3 (Tekrar)
# ReLU aktivasyonunun üstünlüğünü doğrulama

# Yeni ReLU model örneği
model_3 = MNistClassifier_3()

# Eğitim ve performans analizi
# ReLU'nun modern derin öğrenmedeki avantajlarını gözlemleyeceğiz
train_loss, validation_loss = train_model(model_3, data_loader_train, data_loader_test)

# ReLU modelin optimizasyon sürecini görselleştir
# En iyi performans ve hızlı yakınsama bekliyoruz
plot_loss(train_loss, validation_loss)

# Sonuç: Üç modelin karşılaştırmalı analizi tamamlandı
# Model 1: Lineer (basit, sınırlı)
# Model 2: Sigmoid (geleneksel, gradyan kaybı riski)
# Model 3: ReLU (modern, optimize edilmiş)